# Week 15: other transforms and the time-series project

Goals: the Radon transform, the Hough transform, the Zak transform, and a multiscale analysis of a price series.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import edges, images, timeseries, transforms

## Radon transform and filtered back-projection

In [ ]:
size = 128
rows, cols = np.mgrid[:size, :size] - size / 2
phantom = ((rows / 40) ** 2 + (cols / 25) ** 2 <= 1).astype(float)
phantom[50:60, 55:75] = 0.5
sinogram = transforms.radon(phantom)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
images.show_image(phantom, "phantom", ax=axes[0], scaled=True)
images.show_image(sinogram, "sinogram", ax=axes[1], scaled=True)
images.show_image(
    transforms.inverse_radon(sinogram), "back-projection", ax=axes[2], scaled=True
)
plt.show()

## Hough transform of Canny edges

In [ ]:
checker = images.load_test_image("checkerboard")
edge_map = edges.canny(checker, 20, 50)
accumulator, angles, rhos = transforms.hough_lines(edge_map)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
images.show_image(edge_map, "edges", ax=axes[0], scaled=True)
axes[1].imshow(accumulator, aspect="auto", extent=(-90, 90, rhos[-1], rhos[0]))
axes[1].set_xlabel("theta [deg]")
axes[1].set_ylabel("rho")
plt.show()
transforms.strongest_lines(accumulator, angles, rhos, 4)

## Zak transform

In [ ]:
x = np.cos(2 * np.pi * 3 * np.arange(64) / 64)
z = transforms.zak(x, period=8)
z.shape, np.abs(transforms.inverse_zak(z) - x).max()

## Project: a price series across scales

With daily data the coarse components describe years, the middle ones months and the finest days.

In [ ]:
prices = timeseries.synthetic_prices(2048, rng=0)
components = timeseries.multiscale_components(np.log(prices), "db4", 6)
groups = {"large": [0, 1], "medium": [2, 3, 4], "small": [5, 6]}
scales = timeseries.group_scales(components, groups)
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
axes[0].plot(np.log(prices))
axes[0].set_title("log price")
for ax, (name, component) in zip(axes[1:], scales.items(), strict=True):
    ax.plot(component)
    ax.set_title(name)
plt.show()

In [ ]:
other = timeseries.synthetic_prices(2048, rng=1)
returns_a = np.diff(np.log(prices), prepend=np.log(prices[0]))
returns_b = 0.5 * returns_a + np.diff(np.log(other), prepend=np.log(other[0]))
np.round(timeseries.scale_correlations(returns_a, returns_b, "db4", 6), 3)

Project: replace the synthetic series with real daily prices (for example two commodities) and compare their correlation on every scale.